In [3]:
import re
import pandas as pd

In [49]:
def extract_gender(biography, name):
    if not biography or pd.isna(biography):
        return "Unknown"
    
    bio_lower = biography.lower()
    
    # check for explicit gender statement first
    if re.search(r'gender:\s*male', bio_lower):
        return "Male"
    if re.search(r'gender:\s*female', bio_lower):
        return "Female"
    
    # otherwise, count pronouns
    male_pronouns = len(re.findall(r'\b(he|him|his)\b', bio_lower))
    female_pronouns = len(re.findall(r'\b(she|her|hers)\b', bio_lower))
    
    if male_pronouns > female_pronouns:
        return "Male"
    elif female_pronouns > male_pronouns:
        return "Female"
    
    return "Unknown"

In [52]:
# Test it
results_df = pd.read_csv("character_bios_final.csv")

results_df["gender"] = results_df.apply(
    lambda row: extract_gender(row["biography"], row["name"]), axis=1
)

In [53]:
!pip install nltk

In [54]:
# use nltk to extract adjectives from characters' bios to get list of personality traits
import nltk
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk import pos_tag, word_tokenize
from collections import Counter

def extract_adjectives(biography):
    if not biography or pd.isna(biography):
        return []
    tokens = word_tokenize(biography)
    tagged = pos_tag(tokens)
    adjectives = [word.lower() for word, tag in tagged 
                  if tag in ("JJ", "JJR", "JJS")
                  and len(word) > 2]
    return list(set(adjectives))

# Test on a few first
for _, row in results_df.head(10).iterrows():
    adjs = extract_adjectives(row["biography"])
    print(f"{row['name']}: {adjs}\n")

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/marianama/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to /Users/marianama/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/marianama/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Gintoki Sakata: ['most', 'latest', 'such', 'reckless', 'impressive', 'oppressive', 'dentist', 'other', 'old', 'second', 'fearsome', 'odd', 'several', 'great', 'unmatched', 'pachinko', 'jump', 'sweet', 'only', 'big', 'similar', 'initial', 'due', 'major', 'white', 'famous', 'female']

Kagura: []

Shinpachi Shimura: ['competent', 'comedian', '21-22', 'meek', 'leo', 'spilled']

Kotarou Katsura: ['japanese', 'such', 'first', 'left', 'violent', 'wanted', 'other', 'historical', 'important', 'odd', 'outdated', 'dramatic', 'skull', 'past', 'right', 'uncanny', 'similar', 'initial', 'metallic', 'due', 'subsequent', 'former', 'same', 'terrorist', 'complete', 'english']

Toushirou Hijikata: ['right-handed', 'such', 'rival', 'hot-tempered', 'taurus', 'cold', 'famed', 'trouble-maker', 'demonic', 'close', 'popular', 'loyal', 'fearsome', 'constant', 'emotional', 'male', 'compassionate', 'big', 'many', 'human', 'disciplinary', 'good']

Sougo Okita: ['sakura-blossom', 'japanese', 'original', 'unfinished'

In [55]:
from collections import Counter

# extract adjectives from all bios
results_df["all_adjectives"] = results_df["biography"].apply(extract_adjectives)

# get most common across entire dataset
all_adj = []
for adjs in results_df["all_adjectives"].dropna():
    all_adj.extend(adjs)

top_adj = Counter(all_adj).most_common(500)
for adj, count in top_adj:
    print(f"{adj}: {count}")

KeyboardInterrupt: 

In [56]:
# personality adjectives from the list
personality_adjectives = {
    # stereotypically "positive" traits
    "friendly", "cheerful", "kind-hearted", "gentle", "calm", "loyal", "polite", "nice", "kind",
    "honest", "confident", "energetic", "optimistic", "playful", "carefree",
    "innocent", "humble", "reliable", "responsible", "caring", "supportive",
    "helpful", "determined", "passionate", "enthusiastic", "outgoing", "charismatic",
    "mature", "independent", "perceptive", "observant", "thoughtful", "logical",
    "intelligent", "smart", "bright", "capable", "talented", "athletic", "skilled",
    "cool", "elegant", "noble", "protective", "peaceful", "powerful", "pure", "good-natured",
    "popular", "beautiful", "sweet", "sharp", "innocent", "attractive", "talented", "playful",
    "level-headed", "soft-spoken", "strong-willed", "strong", "laid-back", "easygoing", "mature", 
    "successful", "charismatic", "curious", "trusted", "knowledgeable",
    
    # stereotypically "negative" traits
    "arrogant", "selfish", "shy", "timid", "lazy", "naive", "clumsy", "competitive",
    "stubborn", "aggressive", "violent", "cruel", "ruthless", "manipulative",
    "sadistic", "jealous", "childish", "reckless", "impulsive", "perverted",
    "eccentric", "weird", "obsessive", "cynical", "sarcastic", "aloof", 
    "stoic", "serious", "strict", "harsh", "rude", "rebellious", "competitive",
    "proud", "cowardly", "nervous", "awkward", "sensitive", "emotional", "dramatic",
    "mischievous", "wild", "crazy", "mad", "hostile", "fierce", "brutal",
    "mysterious", "secretive", "reserved", "introverted", "tomboyish", "timid",
    "short-tempered", "tsundere", "oblivious", "cautious", "suspicious", "sensitive",
    "reluctant", "loud", "talkative", "bold", "blunt", "direct", "weak", "angry", "oblivious"
}

In [57]:
# Function to extract specified traits from each character's biography
def extract_personality_traits(biography):
    if not biography or pd.isna(biography):
        return []
    
    bio_lower = biography.lower()
    return [trait for trait in personality_adjectives 
            if re.search(rf'\b{trait}\b', bio_lower)]

# Apply to dataset
results_df["personality_traits"] = results_df["biography"].apply(extract_personality_traits)

# Check results
for _, row in results_df.head(10).iterrows():
    print(f"{row['name']}: {row['personality_traits']}")

# See how many characters have at least one trait
has_traits = results_df["personality_traits"].apply(len) > 0
print(f"\nCharacters with at least one trait: {has_traits.sum()} ({has_traits.mean():.1%})")

Gintoki Sakata: ['reckless', 'sweet']
Kagura: []
Shinpachi Shimura: []
Kotarou Katsura: ['violent', 'dramatic']
Toushirou Hijikata: ['loyal', 'emotional', 'popular']
Sougo Okita: ['childish']
Shinsuke Takasugi: ['skilled']
Taizou Hasegawa: ['successful']
Tsukuyo: ['ruthless']
Kamui: []

Characters with at least one trait: 4789 (37.5%)


In [59]:
# Test on specific character to see output
bio = results_df[results_df["mal_id"] == 155904.0]["biography"].values[0]
print(bio)
print(extract_personality_traits(bio))

Age: 17
Birth place: Rokushoukan
Hair color: Black (WN), Dark Green (LN)
Eye color: (Black (WN), Purple (LN)
Height: 153 cm (5'0")

Maomao became a maidservant to the Emperor's harem after being kidnapped and sold. She used to live with her father as a pharmacist in the red light district and is very knowledgeable about medicine and poisons but isn't very interested in other humans.

She is skinny and short, and her plain face is covered with freckles.

AbilitiesMaomao's name is derived from 猫 (māo), the Chinese word for cat. This reinforces the fact that she is often described as a cat or cat-like.


(Source: The Apothecary Diaries Wiki, edited)
['knowledgeable']


In [60]:
# Now check bias / ratio of female to male personalitys extracted
results_df["has_traits"] = results_df["personality_traits"].apply(len) > 0

print("Trait extraction rate by gender:")
print(results_df.groupby("gender")["has_traits"].mean())

print("\nGender distribution overall:")
print(results_df["gender"].value_counts())

Trait extraction rate by gender:
gender
Female     0.558942
Male       0.549083
Unknown    0.046089
Name: has_traits, dtype: float64

Gender distribution overall:
gender
Unknown    4513
Female     4233
Male       4034
Name: count, dtype: int64


In [62]:
print(results_df[results_df["name"] == "Tooru Honda"]["personality_traits"].values[0])

['optimistic', 'polite', 'kind']


In [63]:
# Save results to csv
results_df.to_csv("character_personalities_final.csv", index=False)